In [ ]:
import os
import sys

# 実行環境の確認
if "google.colab" in sys.modules:
    print("✓ Running in Google Colab")
else:
    print("✗ Running locally (not Colab)")

print(f"Python version: {sys.version}")
print(f"Executable: {sys.executable}")
print(f"Working Directory: {os.getcwd()}")

In [ ]:
# Colab で実行している場合、リポジトリをクローンする
#!git clone -b master https://github.com/oreilly-japan/deep-learning-from-scratch-2.git
#%cd deep-learning-from-scratch-2
#sys.path.append('.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from dataset import sequence
from common.trainer import Trainer
from common.optimizer import Adam
from common.time_layers import *
from ch07.seq2seq import Encoder
from ch07.seq2seq import Seq2seq
from ch08.attention_layer import TimeAttention

# 第8章 Attention

本章では、前章で学んだ seq2seq を発展させた Attention (注意機構) という強力な技術を学ぶ。Attention は、必要な情報だけに「注意」を向けさせることで、これまでのモデルが抱えていた根本的な問題を解決する。  

## 8.1 Attention の仕組み

### 8.1.1 seq2seq の問題点
従来の Encoder-Decoder モデルでは、Encoder が入力文の長さに係わらず、常に固定長のベクトルに情報を押し込んでいた。これにより、長い文章が入力された際、必要な情報がベクトルから洩れてしまう (情報欠落) という限界があった。  

cf. 図 8-1 (p326)  

### 8.1.2 Encoder の改良
固定長という制約をなくすため、Encoder の各時刻の隠れ状態ベクトルをすべて利用できるように改良する。  

* 各時刻の隠れ状態には、その直前に入力された単語の情報が多く含まれている。  

* Encoder は入力単語数と同じだけのベクトル集合 `hs` を出力する。  

cf. 図 8-2 (p327)  

### 8.1.3 Decoder の改良 (Weight Sum)
Decoder では、Encoder から受け取った `hs` の中から、その時々で必要な情報 (単語) だけを選び出す仕組みを導入する。  

1. アライメント：  
    翻訳先の単語と対応関係にある翻訳元の単語の情報を抽出すること。

2. 重み付き和：  
    「選ぶ」操作を微分可能にするため、各単語の「重要度」を表す重み `a` を計算し、ベクトル `hs` との重み付き和によってコンテキストベクトル `c` を求める。

In [ ]:
class WeightSum:
    def __init__(self):
        self.params, self.grads = [], []
        self.cache = None

    def forward(self, hs, a):
        """
        hs: Encoderの隠れ状態 (N, T, H)
        a: 各単語への重み (N, T)
        """
        N, T, H = hs.shape
        # 重み a を (N, T, 1) に整形してブロードキャストで掛け合わせる
        ar = a.reshape(N, T, 1)
        t = hs * ar
        # 時間軸 (T) 方向に合計
        c = np.sum(t, axis=1)

        self.cache = (hs, ar)
        return c

    def backward(self, dc):
        hs, ar = self.cache
        N, T, H = hs.shape

        dt = dc.reshape(N, 1, H).repeat(T, axis=1)
        dhs = dt * ar
        dar = dt * hs
        da = np.sum(dar, axis=2)

        return dhs, da

### 8.1.4 Attention Weight の算出
単語の重み `a` は、Decoder の LSTM の隠れ状態 `h` と、Encoder の各単語ベクトル `hs` の「類似度（内積）」によって求める。  

#### Attention の処理フローを追跡する
同一種類のパラメータを均一な値 (2 または 3) にして、計算の流れを確認する。  

cf. 図 8-15 (p341)  

In [ ]:
# パラメータ設定
T = 3  # 文の長さ
H = 2  # ベクトルの次元
N = 1  # バッチサイズ

# Encoderの出力 hs (すべての要素を 2 で埋める)
hs = np.full((N, T, H), 2.0)
# Decoderの現在の隠れ状態 h (すべての要素を 3 で埋める)
h = np.full((N, H), 3.0)

print(f"--- 模擬パラメータ ---")
print(f"hs (Encoder) :\n{hs}\n")
print(f"h  (Decoder) :\n{h}")

In [ ]:
# 1. スコア算出 (内積)
# h を hs と同じ形状に拡張
hr = h.reshape(N, 1, H)
score = np.sum(hs * hr, axis=2) 
print(f"\n1. 類似度スコア (hs * h の内積):\n{score}  # 2*3 + 2*3 = 12")

In [ ]:
# 2. 正規化 (Softmax)
# exp(12) / (exp(12)*3) = 1/3
def softmax(x):
    x = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

a = softmax(score)
print(f"\n2. 正規化された重み a (Softmax):\n{a}  # すべて同じスコアなら均等配分")

In [ ]:
# 3. コンテキストベクトル c (重み付き和)
c = np.sum(hs * a.reshape(N, T, 1), axis=1)
print(f"\n3. コンテキストベクトル c:\n{c}  # 2*(1/3) + 2*(1/3) + 2*(1/3) = 2.0")

### 8.1.5 Attention レイヤの統合
これら「重みの算出」と「重み付き和」をまとめたものが Attention レイヤである。  

cf. 図 8-16 (p343)  

### この節のまとめ
* 固定長ベクトルの問題：長い文章での情報欠落を解決するために、すべての隠れ状態 `hs` を出力する。

* Attention の仕組み：Decoder の状態 `h` と `hs` の内積 (類似度) を計算し、Softmax で重みを決定する。

* 微分可能：「選び出す」操作を重み付き和という計算に置き換えることで、全体を誤差逆伝播法で学習可能にしている。

## 8.2 Attention 付き seq2seq の実装

Attention を組み込んだ seq2seq は、以下の 3 つのクラスで構成される。  

1. AttentionEncoder：  
    すべての隠れ状態 `hs` を出力する。  

2. AttentionDecoder：  
    Attention レイヤを持ち、コンテキストベクトルを利用する。

3. AttentionSeq2seq：  
    上記 2 つを統合する。

### 8.2.1 Encoder の実装
前章の `Encoder` は最後の隠れ状態だけを返していたが、`AttentionEncoder` ではすべての時刻の隠れ状態 `hs` を返す。  

In [ ]:
class AttentionEncoder(Encoder):
    def forward(self, xs):
        xs = self.embed.forward(xs)
        hs = self.lstm.forward(xs)
        return hs

    def backward(self, dhs):
        dout = self.lstm.backward(dhs)
        dout = self.embed.backward(dout)
        return dout

### 8.2.2 Decoder の実装
`AttentionDecoder` は、LSTM の出力 (`dec_hs`) と Encoder の出力 (`enc_hs`) を `TimeAttention` レイヤに入力し、得られたコンテキストベクトルを Affine レイヤに渡す。  

cf. 図 8-21 (p349)  

In [ ]:
class AttentionDecoder:
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        embed_W = (rn(V, D) / 100).astype('f')
        lstm_Wx = (rn(D, 4 * H) / np.sqrt(D)).astype('f')
        lstm_Wh = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm_b = np.zeros(4 * H).astype('f')
        # Affineレイヤには「コンテキストベクトル」と「隠れ状態」の2つが入力されるため、入力次元は 2*H になる
        affine_W = (rn(2*H, V) / np.sqrt(2*H)).astype('f')
        affine_b = np.zeros(V).astype('f')

        self.embed = TimeEmbedding(embed_W)
        self.lstm = TimeLSTM(lstm_Wx, lstm_Wh, lstm_b, stateful=True)
        self.attention = TimeAttention()
        self.affine = TimeAffine(affine_W, affine_b)

        self.params, self.grads = [], []
        for layer in [self.embed, self.lstm, self.attention, self.affine]:
            self.params += layer.params
            self.grads += layer.grads

    def forward(self, xs, enc_hs):
        h = enc_hs[:, -1]
        self.lstm.set_state(h)

        out = self.embed.forward(xs)
        dec_hs = self.lstm.forward(out)
        c = self.attention.forward(enc_hs, dec_hs) # Attentionでコンテキストベクトルを算出
        out = np.concatenate((c, dec_hs), axis=2) # コンテキストと隠れ状態を結合
        score = self.affine.forward(out)

        return score

    def backward(self, dscore):
        dout = self.affine.backward(dscore)
        N, T, H2 = dout.shape
        H = H2 // 2

        dc, ddec_hs0 = dout[:,:,:H], dout[:,:,H:]
        denc_hs, ddec_hs1 = self.attention.backward(dc)
        ddec_hs = ddec_hs0 + ddec_hs1
        dout = self.lstm.backward(ddec_hs)
        dh = self.lstm.dh
        denc_hs[:, -1] += dh
        self.embed.backward(dout)

        return denc_hs

### Attention 付き Decoder の処理を追跡する
コンテキストベクトル `c` と LSTM の出力 `h` がどのように結合され、Affine レイヤでスコアになるかの流れを、均一なパラメータでシミュレーションする。  

In [ ]:
# パラメータ設定
H = 2  # ベクトルの次元
V = 3  # 語彙数
N = 1  # バッチサイズ

# 1. Attentionの出力 c (すべての要素を 2 で埋める)
c = np.full((N, H), 2.0)
# 2. DecoderのLSTMの出力 h (すべての要素を 3 で埋める)
h = np.full((N, H), 3.0)

print(f"--- Decoder 内部処理の追跡 ---")
print(f"コンテキストベクトル c : {c}")
print(f"LSTM 隠れ状態 h       : {h}")

In [ ]:
# 3. 結合 (concatenate)
# 横方向に連結され、形状は (1, 4) になる
out = np.concatenate((c, h), axis=1)
print(f"\n3. 結合後のベクトル (c と h を連結):\n{out}")

In [ ]:
# 4. Affine変換
# 重み W_affine も分かりやすく 1.0 で均一埋め
W_affine = np.full((2 * H, V), 1.0)
score = np.dot(out, W_affine)

print(f"\n4. スコア算出 (out * W_affine):")
print(f"W_affine (2H x V):\n{W_affine}\n")
print(f"最終スコア:\n{score}  # (2+2+3+3)*1.0 = 10.0")

### 8.2.3 AttentionSeq2seq クラス
最後に、Encoder と Decoder を統合する。  

In [ ]:
class AttentionSeq2seq(Seq2seq):
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        args = vocab_size, wordvec_size, hidden_size
        self.encoder = AttentionEncoder(*args)
        self.decoder = AttentionDecoder(*args)
        self.softmax = TimeSoftmaxWithLoss()

        self.params = self.encoder.params + self.decoder.params
        self.grads = self.encoder.grads + self.decoder.grads

### テンソルの形状 (Shape) の変化を確認
複雑なレイヤを組む際は、データの形状がどう変わっていくかを把握することが重要。  

In [ ]:
# 形状の確認用
N, T, H, V = 1, 5, 2, 10
enc_hs = np.random.randn(N, T, H)
dec_xs = np.random.randint(0, V, (N, T))

model = AttentionSeq2seq(V, 10, H)
score = model.decoder.forward(dec_xs, enc_hs)

print(f"Encoder の出力 hs の形状    : {enc_hs.shape}  # (N, T, H)\n")
print(f"Decoder のスコア出力の形状   : {score.shape}  # (N, T, V)")

## 8.3 Attention の評価

### 8.3.1 日付フォーマットの変換問題
ここでは、様々な形式で書かれた日付（例：「september 27, 1994」）を、標準的な形式（「1994-09-27」）へ変換するタスクに取り組みます。  

* 問題の難しさ：  
    入力には「Sep 27, 1994」や「27/9/94」など多様なバリエーションがあり、変換ルールを人手ですべて書くのは困難である。

* データセット：  
    50,000 個の学習データが含まれる `date.txt` を使用する。

cf. 図 8-22 (p352)  

### 8.3.2 Attention 付き seq2seq の学習
日付変換用データセットを対象に学習を行う。ここでは前章のテクニックである「入力文の反転 (Reverse)」も併用する。  

In [ ]:
# 1. データの読み込み
(x_train, t_train), (x_test, t_test) = sequence.load_data('date.txt')
char_to_id, id_to_char = sequence.get_vocab()

# 入力文を反転 (Reverse)
x_train, x_test = x_train[:, ::-1], x_test[:, ::-1]

In [ ]:
# 2. ハイパーパラメータの設定
vocab_size = len(char_to_id)
wordvec_size = 16
hidden_size = 256
batch_size = 128
max_epoch = 5
max_grad = 5.0

model = AttentionSeq2seq(vocab_size, wordvec_size, hidden_size)
optimizer = Adam()
trainer = Trainer(model, optimizer)

In [ ]:
# 3. 学習開始
acc_list = []
for epoch in range(max_epoch):
    trainer.fit(
        x_train, t_train,
        max_epoch=1,
        batch_size=batch_size,
        max_grad=max_grad
    )

# 学習結果の保存
model.save_params()

### 8.3.3 Attention の可視化
Attention レイヤが「どの入力単語」に「どれだけの重み」を置いているかを可視化する。これにより、ブラックボックスになりがちなディープラーニングの判断根拠を人間が理解できるようになる。  

#### Attention の重みをマッピング
行列の値を簡略化して、重みがどのように「注意」として現れるかを可視化する。  

cf. 図 8-27 (p358)  

#### 可視化からわかること
* 対応関係の学習：  
    モデルは「AUGUST」が「08月」に対応することを、明示的なルールなしにデータからのみ学習している。  

* 人間による理解：  
    出力された「1983」が入力の「1983」に基づいていることが一目でわかり、モデルが正しく動作している確証が得られる。  

### この節のまとめ

* 圧倒的な学習スピード：  
    シンプルな seq2seq が苦戦する問題でも、Attention を使うことで極めて短期間に収束する。  

* Peeky モデルとの比較：  
    前章で学んだ Peeky モデルも優秀ですが、学習の速さと将来的な複雑なタスクへの適応力では Attention が勝る。  

* 解釈性 (Explainability)：  
    Attention の重みを可視化することで、モデルが「なぜその答えを出したか」を人間が視覚的に確認できるようになる。  

## 8.4 Attention に関する残りのテーマ

### 8.4.1 双方向 RNN (Bidirectional RNN)
これまでの Encoder は、文章を左から右へ処理していた。しかし、文章の文脈は「後ろ」から決まることもある。双方向 LSTM を用いることで、各単語のベクトルに左右両方向からの情報を集約させることができる。  

* 仕組み：  
    順方向 (左から右) の LSTM と、逆方向 (右から左) の LSTM の 2 つを用意し、各時刻の出力を連結 (concatenate) する。  

* メリット：  
    各単語のベクトルが、その周囲の情報をバランスよく含んだ豊かな表現になる。  

cf. 図 8-30 (p361)  

#### 双方向 RNN の出力 (連結処理)
順方向と逆方向の隠れ状態がどのように合体するかを確認する。  

In [ ]:
# パラメータ設定
N = 1  # バッチサイズ
H = 2  # 各LSTMの隠れ状態の次元

# 1. 順方向 LSTM のある時刻の出力 (すべて 2 で埋める)
h_forward = np.full((N, H), 2.0)
# 2. 逆方向 LSTM の同じ時刻の出力 (すべて 3 で埋める)
h_backward = np.full((N, H), 3.0)

print(f"順方向の出力 h_f: {h_forward}")
print(f"逆方向の出力 h_b: {h_backward}")

In [ ]:
# 3. 連結 (concatenate)
h_bi = np.concatenate((h_forward, h_backward), axis=1)

print(f"\n連結後の隠れ状態 (双方向出力):")
print(f"{h_bi}  # 形状は (N, 2*H) に拡張される")

### 8.4.2 Attention レイヤの使用方法
Attention レイヤ (コンテキストベクトル) をどこで使うかには、いくつかのバリエーションがある。  

1. Affine レイヤへの入力：  
    実装が単純でモジュール化しやすい。  

2. 次時刻の LSTM への入力：  
    コンテキストベクトルの情報を LSTM 自体が直接利用できる構成である。  

cf. 図 8-31 (p363)  

### 8.4.3 seq2seq の深層化と skip コネクション
複雑な翻訳タスクでは、LSTM を多層に重ねることで表現力を高めます。その際、勾配消失を防ぐために skip コネクション (残差接続) が有効である。  

* skip コネクション：  
    層をまたいで「線をつなぐ」だけの単純な仕組みである。  

* ポイント：  
    接続部では出力を加算する。加算は逆伝播時に勾配をそのまま流すため、深い層まで学習が進みやすくなる。  

cf. 図 8-34 (p366)  

### この節のまとめ
* 双方向 RNN：  
    文章を前後から読むことで、各単語のベクトル表現をより豊かにする。  

* 配置の多様性：  
    コンテキストベクトルを次時刻の入力に使うなどの工夫も存在する。  

* 深層化と skip コネクション：  
    性能向上のために多層化する際は、加算による skip コネクションで勾配を保護することが不可欠である。  

## 8.5 Attention の応用

### 8.5.1 Google Neural Machine Translation (GNMT)
Google 翻訳は、2016 年に従来の統計ベースの手法から、ディープラーニングベースの GNMT へと移行した。その基本構造は、本章で実装した「Attention 付き seq2seq」だが、精度向上のために以下の改良が加えられている。  

- 多層化：  
    LSTM レイヤを 8 層重ねて表現力を高めている。  

- 双方向 LSTM：  
    Encoder の 1 層目のみ双方向 LSTM を採用し、文脈を豊かに捉える。  

- Skip コネクション：  
    勾配消失を防ぐために層をまたぐ接続を導入している。  

- 分散学習：  
    多数の GPU を用いた並列処理で学習を高速化している。  

cf. 図 8-35 (p368)  

### 8.5.2 Transformer
RNN は時間方向に逐次計算を行うため、GPU による並列化が難しいというボトルネックがある。これを解決するために、「RNN を一切使わず Attention のみで構成する」モデルとして提案されたのが Transformer である。  

- Self-Attention (自己注意)：  
    2 つの異なる時系列間ではなく、「自分自身」の時系列内での対応関係を計算する仕組みである。  

- 並列計算：  
    RNN のようなループがないため、時間方向の計算を一括で行え、学習時間を大幅に短縮できる。  

cf. 図 8-37 (p371)  

### Self-Attention の計算フロー
Transformer のコアとなる Self-Attention の挙動を、均一なパラメータ（2 または 3）でシミュレーションする。自分自身の各要素にどれだけ「注意」を向けるかを計算する。  

In [ ]:
# パラメータ設定
T = 3  # 文の長さ (例: "I", "am", "cat")
H = 2  # ベクトルの次元
N = 1  # バッチサイズ

# 自身の時系列ベクトル hs (すべての要素を 2 で埋める)
hs = np.full((N, T, H), 2.0)

print(f"--- Self-Attention の入力 ---")
print(f"hs (自身のベクトル集合):\n{hs}")

In [ ]:
# 1. スコア算出
# (N, T, H) とそれ自身の行列積を行う
# 各要素のスコア = 2*2 + 2*2 = 8
s = np.matmul(hs, hs.transpose(0, 2, 1))
print(f"\n1. 類似度スコア s (hs と hs の内積):\n{s}")

In [ ]:
# 2. 正規化 (Softmax)
def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

a = softmax(s)
print(f"\n2. 注意の重み a (Softmax):\n{a}")

In [ ]:
# 3. コンテキストベクトル c (重み付き和)
c = np.matmul(a, hs)
print(f"\n3. 出力ベクトル c (情報の再集約):\n{c}")

### 8.5.3 Neural Turing Machine (NTM)
RNN は内部状態として情報を記憶するが、その容量は固定長であり制限がある。NTM は、RNN の外部に「メモリ (記憶装置)」を配置し、Attention を使ってそのメモリを読み書きするモデルである。  

- コントローラ：  
    RNN などで構成され、情報の処理を行う。  

- 読み書き：  
    「どの番地に書き込むか」「どの番地を読み出すか」を、Attention による重み付き和 (微分可能な操作) で決定する。  

- アルゴリズムの学習：  
    ソートやコピーといった「手順」そのものをデータから学習できる能力を持つ。  

### この節のまとめ

- GNMT：  
    実用的な翻訳精度の向上のため、seq2seq に多層化や Skip コネクションを組み合わせている。  

- Transformer：  
    RNN を捨て、Self-Attention を使うことで並列計算と高精度を両立させている。  

- NTM：  
    Attention を「メモリの読み書き」に応用し、NN にコンピュータのような外部記憶能力を与えている。  

# 8.6 まとめ

本章では、時系列データ変換の最終兵器とも言える Attention (注意機構) の仕組みを学び、実装を行った。これまでの seq2seq が抱えていた「固定長ベクトルのボトルネック」を、この Attention によって見事に解消できることを確認した。  

### 本章のまとめ
本章で学んだ重要なポイント：  

- 対応関係の学習：  
    翻訳や音声認識などのタスクでは、入力と出力の時系列データ間に対応関係が存在する。Attention は、この対応関係をデータから自動的に学習する。  

- 仕組みの核心：  
    Attention は、Encoder のすべての隠れ状態 `hs` を利用する。Decoder の状態 `h` と `hs` の類似度 (内積) から「重み」を算出し、その重み付き和によって最適な情報を抽出した「コンテキストベクトル」を出力する。  

- 微分可能性：  
    Attention で使用する演算はすべて微分可能であるため、これまでのモデルと同様に誤差逆伝播法を用いて学習可能である。

- 可視化と解釈性：  
    Attention が算出する重みを可視化することで、モデルが「なぜその答えを出したのか」という入出力の対応関係を人間が視覚的に確認できるようになる。  

- 広範な応用：  
    Attention は、Transformer (RNNを使わないモデル) や NTM (外部メモリの読み書き) など、最先端の多様な研究分野で基盤技術として利用されている。  